# Дашборд конверсий

Визиты и регистрации берутся из API, расходы на рекламу — из `ads.csv`.
Ноутбук считает конверсию по дням и платформам, сохраняет `conversion.json` и `ads.json`
и строит графики в директорию `charts`.

Запуск: `jupyter execute charts_project.ipynb` (переменные `API_URL`, `DATE_BEGIN`, `DATE_END` — в `.env`).

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import requests
import seaborn as sns
from dotenv import load_dotenv

## Настройки из переменных окружения

In [ ]:
load_dotenv()


def get_env(name):
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f'Не задана переменная окружения {name}, добавьте её в .env')
    return value


API_URL = get_env('API_URL').rstrip('/')
DATE_BEGIN = get_env('DATE_BEGIN')
DATE_END = get_env('DATE_END')

CHARTS_DIR = './charts'
os.makedirs(CHARTS_DIR, exist_ok=True)
sns.set_theme(style='whitegrid')

## Загрузка данных из API

In [ ]:
def fetch(route):
    response = requests.get(
        f'{API_URL}/{route}',
        params={'begin': DATE_BEGIN, 'end': DATE_END},
        timeout=60,
    )
    response.raise_for_status()
    records = response.json()
    if not records:
        raise RuntimeError(f'API вернул пустой ответ для /{route} за {DATE_BEGIN} — {DATE_END}')
    frame = pd.DataFrame(records)
    frame['datetime'] = pd.to_datetime(frame['datetime'])
    return frame


visits = fetch('visits')
registrations = fetch('registrations')
visits.describe(include='all')

In [ ]:
registrations.describe(include='all')

## Конверсия по дням и платформам

Боты отсеиваются по слову `bot` в User-Agent до подсчёта. Посетитель мог заходить несколько раз
до регистрации, поэтому для каждого `visit_id` берётся только последний визит.

In [ ]:
def count_by_day_and_platform(frame, column):
    return (
        frame.assign(date_group=frame['datetime'].dt.normalize())
        .groupby(['date_group', 'platform'])
        .size()
        .rename(column)
        .reset_index()
    )


humans = visits[~visits['user_agent'].str.contains('bot', case=False, na=False)]
last_visits = humans.sort_values('datetime').drop_duplicates('visit_id', keep='last')

visits_by_day = count_by_day_and_platform(last_visits, 'visits')
registrations_by_day = count_by_day_and_platform(registrations, 'registrations')

conversion = visits_by_day.merge(registrations_by_day, on=['date_group', 'platform'], how='left')
conversion['registrations'] = conversion['registrations'].fillna(0).astype(int)
conversion['conversion'] = (
    conversion['registrations'] / conversion['visits'].where(conversion['visits'] > 0) * 100
).fillna(0)
conversion = conversion.sort_values(['date_group', 'platform']).reset_index(drop=True)
conversion.to_json('./conversion.json')
conversion.head()

## Реклама

Расходы и кампании из `ads.csv` присоединяются к итогам по дням. Дни без рекламы остаются
с нулевой стоимостью и кампанией `none`.

In [ ]:
ads_raw = pd.read_csv('./ads.csv')
ads_raw['date_group'] = pd.to_datetime(ads_raw['date']).dt.normalize()
ads_by_day = ads_raw.groupby('date_group').agg(
    cost=('cost', 'sum'),
    utm_campaign=('utm_campaign', 'first'),
)

totals = conversion.groupby('date_group')[['visits', 'registrations']].sum()
ads = totals.join(ads_by_day, how='left').reset_index()
ads['cost'] = ads['cost'].fillna(0).astype(int)
ads['utm_campaign'] = ads['utm_campaign'].fillna('none')
ads = ads.sort_values('date_group').reset_index(drop=True)
ads.to_json('./ads.json')
ads.head()

## Графики

In [ ]:
def label_bars(ax, fmt='{:.0f}'):
    for container in ax.containers:
        ax.bar_label(container, labels=[fmt.format(v) for v in container.datavalues], fontsize=7, rotation=90, padding=2)


def save(fig, name):
    fig.tight_layout()
    fig.savefig(f'{CHARTS_DIR}/{name}.png')
    plt.close(fig)


def daily_bar(frame, column, title, ylabel, name, fmt='{:.0f}'):
    fig, ax = plt.subplots(figsize=(18, 6))
    ax.bar(frame['date_group'].dt.strftime('%Y-%m-%d'), frame[column], color='steelblue')
    ax.set_title(title)
    ax.set_xlabel('Дата')
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=90, labelsize=6)
    label_bars(ax, fmt)
    save(fig, name)


def platform_bar(frame, column, title, ylabel, name):
    fig, ax = plt.subplots(figsize=(18, 6))
    pivot = frame.pivot_table(index='date_group', columns='platform', values=column, aggfunc='sum', fill_value=0)
    pivot.index = pivot.index.strftime('%Y-%m-%d')
    pivot.plot(kind='bar', stacked=True, ax=ax, width=0.9)
    ax.set_title(title)
    ax.set_xlabel('Дата')
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=90, labelsize=6)
    ax.legend(title='Платформа')
    for container in ax.containers:
        ax.bar_label(container, label_type='center', fontsize=5)
    save(fig, name)

In [ ]:
daily_bar(ads, 'visits', 'Итоговые визиты по дням', 'Визиты, шт.', 'total_visits')
platform_bar(conversion, 'visits', 'Визиты по платформам', 'Визиты, шт.', 'visits_by_platform')
daily_bar(ads, 'registrations', 'Итоговые регистрации по дням', 'Регистрации, шт.', 'total_registrations')
platform_bar(conversion, 'registrations', 'Регистрации по платформам', 'Регистрации, шт.', 'registrations_by_platform')

In [ ]:
platforms = sorted(conversion['platform'].unique())
fig, axes = plt.subplots(len(platforms), 1, figsize=(18, 4 * len(platforms)), sharex=True, sharey=True)
for ax, platform in zip(axes, platforms):
    part = conversion[conversion['platform'] == platform]
    ax.plot(part['date_group'], part['conversion'], marker='o', markersize=3)
    for date, value in zip(part['date_group'][::7], part['conversion'][::7]):
        ax.annotate(f'{value:.0f}%', (date, value), textcoords='offset points', xytext=(0, 6), ha='center', fontsize=7)
    ax.set_title(f'Конверсия визитов в регистрации: {platform}')
    ax.set_ylabel('Конверсия, %')
axes[-1].set_xlabel('Дата')
save(fig, 'conversion_by_platform')

average = ads.assign(conversion=ads['registrations'] / ads['visits'].where(ads['visits'] > 0) * 100).fillna({'conversion': 0})
fig, ax = plt.subplots(figsize=(18, 6))
ax.plot(average['date_group'], average['conversion'], marker='o', markersize=3)
for date, value in zip(average['date_group'][::7], average['conversion'][::7]):
    ax.annotate(f'{value:.0f}%', (date, value), textcoords='offset points', xytext=(0, 6), ha='center', fontsize=7)
ax.set_title('Средняя конверсия визитов в регистрации по дням')
ax.set_xlabel('Дата')
ax.set_ylabel('Конверсия, %')
save(fig, 'average_conversion')

In [ ]:
daily_bar(ads, 'cost', 'Расходы на рекламу по дням', 'Стоимость, руб.', 'ads_cost')


def campaign_chart(column, title, ylabel, name):
    fig, ax = plt.subplots(figsize=(18, 6))
    ax.plot(ads['date_group'], ads[column], color='black', linewidth=1)
    campaigns = [c for c in ads['utm_campaign'].unique() if c != 'none']
    palette = sns.color_palette('tab10', len(campaigns))
    for color, campaign in zip(palette, campaigns):
        days = ads.loc[ads['utm_campaign'] == campaign, 'date_group']
        ax.axvspan(days.min(), days.max() + pd.Timedelta(days=1), color=color, alpha=0.25, label=campaign)
    mean = ads[column].mean()
    ax.axhline(mean, color='grey', linestyle='--', label=f'Среднее: {mean:.0f}')
    ax.set_title(title)
    ax.set_xlabel('Дата')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8, loc='upper left')
    save(fig, name)


campaign_chart('visits', 'Визиты и рекламные кампании', 'Визиты, шт.', 'visits_with_campaigns')
campaign_chart('registrations', 'Регистрации и рекламные кампании', 'Регистрации, шт.', 'registrations_with_campaigns')